In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from accelerate.test_utils.testing import get_backend
device, n_devices, _ = get_backend()

torch.set_float32_matmul_precision("highest")

## Warmup: Fitting gradients of a known function class

In [2]:
# Define your RFunction model (unchanged)
class RFunction(nn.Module):
    def __init__(self):
        super().__init__()
        self.theta = nn.Parameter(torch.tensor([1.0, 1.0]))  # Initialize parameters

    def forward(self, A):
        return torch.sum(self.theta[0] * torch.exp(self.theta[1] * A))  # Example function R(A, θ)

# For this example, we'll use the same simulated data, but in batches
A_data = torch.randn(500, 5)  # More data points
true_theta = torch.tensor([2.0, 0.5]) 
true_gradients = true_theta[0] * true_theta[1] * torch.exp(true_theta[1] * A_data)  # Simulated ∇R(A_i)
# Make sure to reshape true_gradients to match the batch size

# Create dataset and dataloader
dataset = TensorDataset(A_data, true_gradients)
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Instantiate model
model = RFunction()
optimizer = optim.Adam(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

# Training loop with batches
num_epochs = 30

for epoch in range(num_epochs):
    epoch_loss = 0.0
    batch_count = 0
    
    for A_batch, true_gradients_batch in dataloader:
        optimizer.zero_grad()
        
        # Make inputs require gradients
        A_batch = A_batch.detach().requires_grad_()
        
        # Compute predicted gradient ∇R(A, θ) via autograd
        R_values = model(A_batch)
        gradients = torch.autograd.grad(R_values, A_batch, create_graph=True)[0]
        
        # Compute loss for this batch

        loss = loss_fn(gradients, true_gradients_batch)
        
        # Backpropagate and update θ
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        batch_count += 1
    
    avg_epoch_loss = epoch_loss / batch_count
    print(f"Epoch {epoch}: Average Loss = {avg_epoch_loss:.8f}")

# Learned parameters
print("Learned theta:", model.theta.detach().numpy())

Epoch 0: Average Loss = 0.57228558
Epoch 1: Average Loss = 0.13411601
Epoch 2: Average Loss = 0.09898319
Epoch 3: Average Loss = 0.04005918
Epoch 4: Average Loss = 0.02100140
Epoch 5: Average Loss = 0.00800053
Epoch 6: Average Loss = 0.00486015
Epoch 7: Average Loss = 0.00271549
Epoch 8: Average Loss = 0.00162890
Epoch 9: Average Loss = 0.00092361
Epoch 10: Average Loss = 0.00057844
Epoch 11: Average Loss = 0.00032834
Epoch 12: Average Loss = 0.00018741
Epoch 13: Average Loss = 0.00011018
Epoch 14: Average Loss = 0.00005632
Epoch 15: Average Loss = 0.00003184
Epoch 16: Average Loss = 0.00001690
Epoch 17: Average Loss = 0.00000810
Epoch 18: Average Loss = 0.00000417
Epoch 19: Average Loss = 0.00000223
Epoch 20: Average Loss = 0.00000096
Epoch 21: Average Loss = 0.00000046
Epoch 22: Average Loss = 0.00000021
Epoch 23: Average Loss = 0.00000009
Epoch 24: Average Loss = 0.00000004
Epoch 25: Average Loss = 0.00000002
Epoch 26: Average Loss = 0.00000001
Epoch 27: Average Loss = 0.00000000
Ep